In [1]:
import dask.dataframe as dd
from dask.distributed import Client, progress
import time
import numpy as np

# Setup Dask distributed client for parallel processing at the start
client = Client(memory_limit='8GB')
print(f"Dask dashboard available at: {client.dashboard_link}")
client

/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
/home/artificialstupidity/ttc-bus-delay-prediction/.venv/lib/python3.12/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package tha

Dask dashboard available at: http://127.0.0.1:8787/status


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 16,Total memory: 29.80 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37195,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:41015,Total threads: 4
Dashboard: http://127.0.0.1:34955/status,Memory: 7.45 GiB
Nanny: tcp://127.0.0.1:33903,


2026-03-22 12:06:00,671 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 2151077bb3d9105a6919cb7ffd80b77d initialized by task ('shuffle-transfer-2151077bb3d9105a6919cb7ffd80b77d', 123) executed on worker tcp://127.0.0.1:41279
2026-03-22 12:06:30,321 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 2151077bb3d9105a6919cb7ffd80b77d deactivated due to stimulus 'task-finished-1774160490.320858'
2026-03-22 12:07:29,332 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 2ab6c74701cb3571e44a52c49ea6998a initialized by task ('shuffle-transfer-2ab6c74701cb3571e44a52c49ea6998a', 92) executed on worker tcp://127.0.0.1:41279
2026-03-22 12:08:14,873 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 2ab6c74701cb3571e44a52c49ea6998a deactivated due to stimulus 'task-finished-1774160594.8725216'
2026-03-22 12:08:20,484 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle b1557d62138bd5757b50e6ad65621f3a initialized by task ('shuffle-transfer-b1557d62138b

In [2]:
df = dd.read_csv("./tmp/unfv.csv", 
                 usecols=[
                     "trip_id",
                     "time",
                     "?column?",
                     'stopid',
                     'route',
                     'thrusteet'
                 ],
                 dtype={
                     "trip_id": "object",
                     'stopid': 'object',
                    }
                 )

# df = dd.read_csv("./tmp/unfv.csv",
#                  dtype={'direction': 'float64',
#        'timespent': 'float64',
#        'trip_id': 'object',
#        'stopid': 'object',})

In [3]:
df.head()

,route,trip_id,stopid,thrusteet,time,?column?
0,86,88277020,3803_1,Kennedy Station,2025-01-14 17:39:53,2025-01-14 17:41:00
1,86,88277020,3803_1,Kennedy Station,2025-01-14 17:39:53,2025-01-14 17:41:00
2,86,88277020,3803_1,Kennedy Station,2025-01-14 17:39:53,2025-01-14 17:41:00
3,86,88277020,3803_1,Kennedy Station,2025-01-14 17:39:53,2025-01-14 17:41:00
4,86,88277020,3803_1,Kennedy Station,2025-01-14 17:39:53,2025-01-14 17:41:00


In [4]:
df = df.rename(columns={
    "time": "actual_arrival",
    "?column?": "planned_arrival"
})

In [5]:
df.thrusteet.value_counts().compute()

thrusteet
Gordon Baker Rd                               82307
Harold Evans Cres                             13704
Highland Creek Overpass                      535734
Toryork Dr                                    73204
Fenmar Dr                                     14594
Eglinton Ave East                           5380318
Nova Scotia Ave                               14875
Finch Station                               4591108
Humberline Dr                               1019458
Princes' Blvd                                 27209
Baldoon Rd                                    27626
Humberwood Blvd Loop                         579095
Finch Ave West - St Wilfreds School          487319
Coronation Dr                                152351
Finch Ave West - Finch/Weston Centre         211794
Meadowvale Rd                               1241537
Beechgrove Dr                                 50582
Finch Ave East - St Aidan                    210500
Park Rd (Zoo)                                 82108
La

We will be using route 29 and 29

In [6]:
df = df[df['route'].isin([29, 39])]

In [7]:
# df.shape[0].compute()

In [8]:
# change to python datetime
df['actual_arrival'] = dd.to_datetime(df['actual_arrival'], format='mixed')
df['planned_arrival'] = dd.to_datetime(df['planned_arrival'], format='mixed')

In [9]:
# df.head(5)

In [10]:
## only take data where delay is positive
df = df[df['actual_arrival'] > df['planned_arrival']]

In [11]:
# df.shape[0].compute()

In [12]:
df = df.drop_duplicates(subset=['stopid', 'trip_id', 'actual_arrival', 'planned_arrival', 'route'])
df = df.sort_values('actual_arrival')

In [13]:
# df.head(5)

In [14]:
# df.shape[0].compute()

In [15]:
df['hour'] = df['planned_arrival'].dt.hour
df['day_of_week'] = df['planned_arrival'].dt.dayofweek
df['month'] = df['planned_arrival'].dt.month
df['day_of_year'] = df['planned_arrival'].dt.dayofyear

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['delay_seconds'] = (df['actual_arrival'] - df['planned_arrival']).dt.total_seconds()

In [16]:
df.dtypes

route                      float64
trip_id            string[pyarrow]
stopid             string[pyarrow]
thrusteet          string[pyarrow]
actual_arrival      datetime64[ns]
planned_arrival     datetime64[ns]
hour                         int32
day_of_week                  int32
month                        int32
day_of_year                  int32
hour_sin                   float64
hour_cos                   float64
day_of_week_sin            float64
day_of_week_cos            float64
delay_seconds              float64
dtype: object

In [17]:
# df.head()

In [18]:
# df.head(5)

In [19]:
weather_data = dd.read_csv("../data/raw/weather_yearly/weather_2025_yearly.csv",
                           usecols=['Climate ID', 
                                    'Date/Time (LST)',
                                    'Temp (°C)',
                                    'Dew Point Temp (°C)',
                                    'Rel Hum (%)',
                                    'Wind Spd (km/h)',
                                    'Visibility (km)',
                                    'Stn Press (kPa)'])

weather_data_new = dd.read_csv("en_climate_hourly_ON_6158359_2025_combined.csv",
                           dtype={'Precip. Amount Flag': 'object',
       'Visibility Flag': 'object',
       'Wind Dir Flag': 'object',
       'Wind Spd Flag': 'object'})

In [20]:
weather_data.compute().shape

(8760, 8)

In [21]:
weather_data_new.compute().shape

(8760, 31)

In [22]:
weather_data.isnull().sum().compute()

Climate ID             0
Date/Time (LST)        0
Temp (°C)              1
Dew Point Temp (°C)    1
Rel Hum (%)            1
Wind Spd (km/h)        1
Visibility (km)        1
Stn Press (kPa)        1
dtype: int64

In [23]:
weather_data_new.isna().sum().compute()

Longitude (x)             0
Latitude (y)              0
Station Name              0
Climate ID                0
Date/Time (LST)           0
Year                      0
Month                     0
Day                       0
Time (LST)                0
Flag                   8760
Temp (°C)                20
Temp Flag              8760
Dew Point Temp (°C)      20
Dew Point Temp Flag    8760
Rel Hum (%)              20
Rel Hum Flag           8760
Precip. Amount (mm)      27
Precip. Amount Flag    8753
Wind Dir (10s deg)      554
Wind Dir Flag          8486
Wind Spd (km/h)          35
Wind Spd Flag          8745
Visibility (km)          23
Visibility Flag        8757
Stn Press (kPa)          20
Stn Press Flag         8760
Hmdx                   7368
Hmdx Flag              8760
Wind Chill             6846
Wind Chill Flag        8760
Weather                7521
dtype: int64

In [24]:
weather_data['Precip. Amount (mm)'] = weather_data_new['Precip. Amount (mm)']

In [25]:
weather_data.head()

,Climate ID,Date/Time (LST),Temp (°C),Dew Point Temp (°C),Rel Hum (%),Wind Spd (km/h),Visibility (km),Stn Press (kPa),Precip. Amount (mm)
0,6158731,2025-01-01 00:00,1.6,1.6,100.0,27.0,16.1,97.93,0.5
1,6158731,2025-01-01 01:00,1.6,1.6,100.0,29.0,19.3,97.95,0.2
2,6158731,2025-01-01 02:00,1.4,1.4,100.0,30.0,19.3,97.98,0.2
3,6158731,2025-01-01 03:00,0.8,0.8,100.0,26.0,9.7,98.03,0.0
4,6158731,2025-01-01 04:00,1.4,1.4,100.0,23.0,19.3,98.07,0.0


In [26]:
weather_data.isna().sum().compute()

Climate ID              0
Date/Time (LST)         0
Temp (°C)               1
Dew Point Temp (°C)     1
Rel Hum (%)             1
Wind Spd (km/h)         1
Visibility (km)         1
Stn Press (kPa)         1
Precip. Amount (mm)    27
dtype: int64

In [27]:
weather_data = weather_data.fillna(0)

In [28]:
weather_data.isnull().sum().compute()

Climate ID             0
Date/Time (LST)        0
Temp (°C)              0
Dew Point Temp (°C)    0
Rel Hum (%)            0
Wind Spd (km/h)        0
Visibility (km)        0
Stn Press (kPa)        0
Precip. Amount (mm)    0
dtype: int64

In [29]:
weather_data['Date/Time (LST)'] = dd.to_datetime(weather_data['Date/Time (LST)'], format='mixed')

In [30]:
weather_data.dtypes

Climate ID                    float64
Date/Time (LST)        datetime64[ns]
Temp (°C)                     float64
Dew Point Temp (°C)           float64
Rel Hum (%)                   float64
Wind Spd (km/h)               float64
Visibility (km)               float64
Stn Press (kPa)               float64
Precip. Amount (mm)           float64
dtype: object

In [31]:
weather_data.head()

,Climate ID,Date/Time (LST),Temp (°C),Dew Point Temp (°C),Rel Hum (%),Wind Spd (km/h),Visibility (km),Stn Press (kPa),Precip. Amount (mm)
0,6158731,2025-01-01 00:00:00,1.6,1.6,100.0,27.0,16.1,97.93,0.5
1,6158731,2025-01-01 01:00:00,1.6,1.6,100.0,29.0,19.3,97.95,0.2
2,6158731,2025-01-01 02:00:00,1.4,1.4,100.0,30.0,19.3,97.98,0.2
3,6158731,2025-01-01 03:00:00,0.8,0.8,100.0,26.0,9.7,98.03,0.0
4,6158731,2025-01-01 04:00:00,1.4,1.4,100.0,23.0,19.3,98.07,0.0


In [32]:
weather_data.shape[0].compute()

8760

In [33]:
import pandas as pd
def near_whole_hour(dt):
    if dt.minute >= 30:
        return pd.to_datetime(dt.round('h'))
    else:
        return pd.to_datetime(dt.replace(minute=0, second=0, microsecond=0))

In [34]:
print(near_whole_hour(pd.to_datetime('2025-01-01 00:45:00')))  # Should return 2025-01-01 13:00:00
print(near_whole_hour(pd.to_datetime('2025-01-01 00:15:00')))  # Should return 2025-01-01 12:00:00

2025-01-01 01:00:00
2025-01-01 00:00:00


In [36]:
# Create nearest_whole_hour column by rounding actual_arrival to the nearest hour
df['nearest_whole_hour'] = df['actual_arrival'].apply(near_whole_hour, meta=('actual_arrival', 'datetime64[ns]'))
# df['nearest_whole_hour'] = dd.to_datetime(df['nearest_whole_hour'], format='mixed')

In [37]:
df.dtypes

route                         float64
trip_id               string[pyarrow]
stopid                string[pyarrow]
thrusteet             string[pyarrow]
actual_arrival         datetime64[ns]
planned_arrival        datetime64[ns]
hour                            int32
day_of_week                     int32
month                           int32
day_of_year                     int32
hour_sin                      float64
hour_cos                      float64
day_of_week_sin               float64
day_of_week_cos               float64
delay_seconds                 float64
nearest_whole_hour     datetime64[ns]
dtype: object

In [38]:
df.head(10)

,route,trip_id,stopid,thrusteet,actual_arrival,planned_arrival,hour,day_of_week,month,day_of_year,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,delay_seconds,nearest_whole_hour
78827,29.0,52553080,34800,Princes' Blvd,2025-01-01 00:06:13.000,2025-01-01 00:05:45,0,2,1,1,0.0,1.0,0.974928,-0.222521,28.000,2025-01-01
160825,39.0,76911080,8878,Finch Ave East,2025-01-01 00:07:14.000,2025-01-01 00:06:20,0,2,1,1,0.0,1.0,0.974928,-0.222521,54.000,2025-01-01
78725,29.0,172354080,1361,Dufferin St,2025-01-01 00:07:26.027,2025-01-01 00:06:57,0,2,1,1,0.0,1.0,0.974928,-0.222521,29.027,2025-01-01
160834,39.0,76911080,8258,Finch Ave East,2025-01-01 00:07:29.928,2025-01-01 00:06:54,0,2,1,1,0.0,1.0,0.974928,-0.222521,35.928,2025-01-01
78828,29.0,52553080,2815,Saskatchewan Rd,2025-01-01 00:07:35.995,2025-01-01 00:07:18,0,2,1,1,0.0,1.0,0.974928,-0.222521,17.995,2025-01-01
160843,39.0,76911080,10060,Finch Ave East,2025-01-01 00:07:48.000,2025-01-01 00:07:32,0,2,1,1,0.0,1.0,0.974928,-0.222521,16.000,2025-01-01
78726,29.0,172354080,9023,Dufferin St,2025-01-01 00:07:58.226,2025-01-01 00:07:53,0,2,1,1,0.0,1.0,0.974928,-0.222521,5.226,2025-01-01
78829,29.0,52553080,18388,Dufferin St,2025-01-01 00:08:48.000,2025-01-01 00:08:14,0,2,1,1,0.0,1.0,0.974928,-0.222521,34.000,2025-01-01
78830,29.0,52553080,1332,Dufferin St,2025-01-01 00:09:51.000,2025-01-01 00:09:30,0,2,1,1,0.0,1.0,0.974928,-0.222521,21.000,2025-01-01
85828,29.0,147096080,3880_0,Wilson Station,2025-01-01 00:10:53.198,2025-01-01 00:10:00,0,2,1,1,0.0,1.0,0.974928,-0.222521,53.198,2025-01-01


In [40]:
weather_data.head()

,Climate ID,Date/Time (LST),Temp (°C),Dew Point Temp (°C),Rel Hum (%),Wind Spd (km/h),Visibility (km),Stn Press (kPa),Precip. Amount (mm)
0,6158731,2025-01-01 00:00:00,1.6,1.6,100.0,27.0,16.1,97.93,0.5
1,6158731,2025-01-01 01:00:00,1.6,1.6,100.0,29.0,19.3,97.95,0.2
2,6158731,2025-01-01 02:00:00,1.4,1.4,100.0,30.0,19.3,97.98,0.2
3,6158731,2025-01-01 03:00:00,0.8,0.8,100.0,26.0,9.7,98.03,0.0
4,6158731,2025-01-01 04:00:00,1.4,1.4,100.0,23.0,19.3,98.07,0.0


In [41]:
# Rename weather column to match our nearest_whole_hour for merging
weather_data = weather_data.rename(columns={'Date/Time (LST)': 'nearest_whole_hour'})

In [42]:
weather_data.head()

,Climate ID,nearest_whole_hour,Temp (°C),Dew Point Temp (°C),Rel Hum (%),Wind Spd (km/h),Visibility (km),Stn Press (kPa),Precip. Amount (mm)
0,6158731,2025-01-01 00:00:00,1.6,1.6,100.0,27.0,16.1,97.93,0.5
1,6158731,2025-01-01 01:00:00,1.6,1.6,100.0,29.0,19.3,97.95,0.2
2,6158731,2025-01-01 02:00:00,1.4,1.4,100.0,30.0,19.3,97.98,0.2
3,6158731,2025-01-01 03:00:00,0.8,0.8,100.0,26.0,9.7,98.03,0.0
4,6158731,2025-01-01 04:00:00,1.4,1.4,100.0,23.0,19.3,98.07,0.0


In [43]:
# Merge df with weather_data on nearest_whole_hour
print("Starting merge operation...")
start_time = time.time()

# Perform the merge
df_with_weather = df.merge(
    weather_data,
    on='nearest_whole_hour',
    how='left'
)

merge_time = time.time() - start_time
print(f"Merge operation set up. Time elapsed: {merge_time:.2f} seconds")
print(f"Total partitions in merged dataframe: {df_with_weather.npartitions}")

Starting merge operation...
Merge operation set up. Time elapsed: 0.01 seconds
Total partitions in merged dataframe: 142


In [44]:
# Preview the merged data structure (lazy operation)
df_with_weather.head()

,route,trip_id,stopid,thrusteet,actual_arrival,planned_arrival,hour,day_of_week,month,day_of_year,...,delay_seconds,nearest_whole_hour,Climate ID,Temp (°C),Dew Point Temp (°C),Rel Hum (%),Wind Spd (km/h),Visibility (km),Stn Press (kPa),Precip. Amount (mm)
0,29.0,52553080,34800,Princes' Blvd,2025-01-01 00:06:13.000,2025-01-01 00:05:45,0,2,1,1,...,28.000,2025-01-01,6158731,1.6,1.6,100.0,27.0,16.1,97.93,0.5
1,39.0,76911080,8878,Finch Ave East,2025-01-01 00:07:14.000,2025-01-01 00:06:20,0,2,1,1,...,54.000,2025-01-01,6158731,1.6,1.6,100.0,27.0,16.1,97.93,0.5
2,29.0,172354080,1361,Dufferin St,2025-01-01 00:07:26.027,2025-01-01 00:06:57,0,2,1,1,...,29.027,2025-01-01,6158731,1.6,1.6,100.0,27.0,16.1,97.93,0.5
3,39.0,76911080,8258,Finch Ave East,2025-01-01 00:07:29.928,2025-01-01 00:06:54,0,2,1,1,...,35.928,2025-01-01,6158731,1.6,1.6,100.0,27.0,16.1,97.93,0.5
4,29.0,52553080,2815,Saskatchewan Rd,2025-01-01 00:07:35.995,2025-01-01 00:07:18,0,2,1,1,...,17.995,2025-01-01,6158731,1.6,1.6,100.0,27.0,16.1,97.93,0.5


In [47]:
# Persist the merged dataframe in distributed memory for faster operations
print("Persisting merged dataframe in distributed memory...")
print(f"Processing {df_with_weather.npartitions} partitions in parallel...")
start_time = time.time()

df_with_weather = df_with_weather.persist()

# Wait for computation to complete and show progress
progress(df_with_weather)

persist_time = time.time() - start_time
print(f"\nPersist operation completed in {persist_time:.2f} seconds ({persist_time/60:.2f} minutes)")

Persisting merged dataframe in distributed memory...
Processing 142 partitions in parallel...

Persist operation completed in 0.03 seconds (0.00 minutes)


In [48]:
# Display final columns
print("Final dataframe columns:")
print(df_with_weather.columns.tolist())

Final dataframe columns:
['route', 'trip_id', 'stopid', 'thrusteet', 'actual_arrival', 'planned_arrival', 'hour', 'day_of_week', 'month', 'day_of_year', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'delay_seconds', 'nearest_whole_hour', 'Climate ID', 'Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)', 'Precip. Amount (mm)']


In [49]:
# Drop datetime columns before saving to reduce file size and memory usage
print("Dropping planned_arrival column...")
df_to_save = df_with_weather.drop(columns=['planned_arrival'])
# Save the merged dataframe as parquet with parallel processing
output_path_29_and_39 = "./tmp/unfv_with_weather_route_29_39.parquet"
print(f"Saving merged dataframe to {output_path_29_and_39}...")
print(f"Using parallel processing with {df_to_save.npartitions} partitions...")
save_start = time.time()

# Save to parquet format (Dask will write in parallel)
df_to_save.to_parquet(
    output_path_29_and_39,
    engine='pyarrow',
    compression='snappy',  # Good balance between compression and speed
    write_index=False
)

save_time = time.time() - save_start
print("\n✓ Successfully saved to parquet!")
print(f"Save time: {save_time:.2f} seconds ({save_time/60:.2f} minutes)")
print(f"Output location: {output_path_29_and_39 }")

Dropping planned_arrival column...
Saving merged dataframe to ./tmp/unfv_with_weather_route_29_39.parquet...
Using parallel processing with 142 partitions...

✓ Successfully saved to parquet!
Save time: 66.42 seconds (1.11 minutes)
Output location: ./tmp/unfv_with_weather_route_29_39.parquet


In [50]:
# Verify the saved parquet file
print("Verifying saved parquet file...")
verify_start = time.time()
df_verify = dd.read_parquet(output_path_29_and_39)
verify_rows = df_verify.shape[0].compute()
verify_cols = len(df_verify.columns)
verify_time = time.time() - verify_start
print(f"Verified rows in parquet file: {verify_rows:,}")
print(f"Number of columns: {verify_cols}")
print(f"Columns: {df_verify.columns.tolist()}")
print(f"Verification time: {verify_time:.2f} seconds")
print("\n✓ All data successfully saved and verified!")
print("\nPreview of first few rows:")
print(df_verify.head())

Verifying saved parquet file...
Verified rows in parquet file: 4,244,111
Number of columns: 23
Columns: ['route', 'trip_id', 'stopid', 'thrusteet', 'actual_arrival', 'hour', 'day_of_week', 'month', 'day_of_year', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'delay_seconds', 'nearest_whole_hour', 'Climate ID', 'Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)', 'Precip. Amount (mm)']
Verification time: 0.11 seconds

✓ All data successfully saved and verified!

Preview of first few rows:
   route    trip_id stopid        thrusteet          actual_arrival  hour  \
0   29.0   52553080  34800    Princes' Blvd 2025-01-01 00:06:13.000     0   
1   39.0   76911080   8878   Finch Ave East 2025-01-01 00:07:14.000     0   
2   29.0  172354080   1361      Dufferin St 2025-01-01 00:07:26.027     0   
3   39.0   76911080   8258   Finch Ave East 2025-01-01 00:07:29.928     0   
4   29.0   52553080   2815  Saskatchewan Rd 2025-01

In [51]:
# Check final shape
print("Computing final dataframe shape...")
shape_start = time.time()
final_shape = df_with_weather.shape[0].compute()
shape_time = time.time() - shape_start
print(f"Total rows: {final_shape:,}")
print(f"Time taken: {shape_time:.2f} seconds")

Computing final dataframe shape...
Total rows: 4,244,111
Time taken: 0.19 seconds
